In [1]:
import os
import re
import pandas as pd
from datetime import datetime
import json
from jobspy import scrape_jobs
from apify_client import ApifyClient
from flashtext import KeywordProcessor
from dotenv import load_dotenv

In [2]:
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [3]:
load_dotenv()
FILE_NAME = 'my_job_tracker.xlsx'
APIFY_TOKEN = os.getenv("APIFY_TOKEN")

In [4]:
SEARCH_TERMS = ["data engineer", "data scientist", "machine learning engineer", "data analyst", "AI engineer"]
RESULTS_PER_TERM = 10

In [5]:
# def normalize_location(loc_str):
#     if not isinstance(loc_str, str): return "Singapore"
#     if 'singapore' in loc_str.lower() or 'sg' in loc_str.lower(): return "Singapore"
#     return loc_str.strip()
    
def extract_years_exp(description):
    if not isinstance(description, str): return None
    sentences = re.split(r'[\n\.]', description)
    for sentence in sentences:
        if 'year' in sentence.lower() and ('experience' in sentence.lower() or 'required' in sentence.lower()):
            match = re.search(r'(\d+)\s*(?:-\s*\d+)?\s*\+?\s*years?', sentence, re.IGNORECASE)
            if match: return int(match.group(1))
    return None

def format_jobspy_salary(row):
    min_amt, max_amt = row.get('min_amount'), row.get('max_amount')
    curr = row.get('currency', 'Unknown')
    interval = row.get('interval', '')
    if pd.notna(min_amt) and pd.notna(max_amt):
        return f"{curr} {int(min_amt):,} - {int(max_amt):,} ({interval})".strip()
    elif pd.notna(min_amt):
        return f"{curr} minimum amount: {int(min_amt):,} ({interval})".strip()
    return ""

def get_work_arrangement(row):
    if row.get('is_remote') == True: return 'Remote'
    wfh = str(row.get('work_from_home_type', '')).lower()
    if 'hybrid' in wfh: return 'Hybrid'
    return 'On-site/Unspecified'

def clean_emails(x):
    if isinstance(x, list):
        emails = x
    elif isinstance(x, str):
        emails = x.split(',')
    else:
        emails = []
    emails = list(dict.fromkeys(e.strip() for e in emails if e and e.strip()))  # dedupe, preserve order
    return ', '.join(emails)

## Jobspy

In [6]:
indeed_frames, linkedin_frames = [], []

for term in SEARCH_TERMS:
    print(f"Scraping Indeed for '{term}'...")
    indeed_frames.append(scrape_jobs(
        site_name=["indeed"],
        search_term=term,
        location="Singapore",
        country_indeed='singapore',
        results_wanted=RESULTS_PER_TERM,
    ))

    print(f"Scraping LinkedIn for '{term}'...")
    linkedin_frames.append(scrape_jobs(
        site_name=["linkedin"],
        search_term=term,
        location="Singapore",
        results_wanted=RESULTS_PER_TERM,
        linkedin_fetch_description=True,
    ))

jobspy_df = pd.concat(indeed_frames + linkedin_frames, ignore_index=True)


Scraping Indeed for 'data engineer'...
Scraping LinkedIn for 'data engineer'...


2026-09-04 11:17:08,999 - INFO - JobSpy:Linkedin - finished scraping


Scraping Indeed for 'data scientist'...
Scraping LinkedIn for 'data scientist'...
Scraping Indeed for 'machine learning engineer'...
Scraping LinkedIn for 'machine learning engineer'...
Scraping Indeed for 'data analyst'...
Scraping LinkedIn for 'data analyst'...
Scraping Indeed for 'AI engineer'...
Scraping LinkedIn for 'AI engineer'...


In [ ]:
# jobspy_df = pd.read_csv("./jobspy_output.csv")

In [7]:
jobspy_df.info(max_cols=None)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id                     100 non-null    object
 1   site                   100 non-null    object
 2   job_url                100 non-null    object
 3   job_url_direct         50 non-null     object
 4   title                  100 non-null    object
 5   company                100 non-null    object
 6   location               100 non-null    object
 7   date_posted            97 non-null     object
 8   job_type               84 non-null     object
 9   salary_source          0 non-null      object
 10  interval               0 non-null      object
 11  min_amount             0 non-null      object
 12  max_amount             0 non-null      object
 13  currency               0 non-null      object
 14  is_remote              100 non-null    bool  
 15  job_level              5

In [17]:
print(jobspy_df.columns)

Index(['id', 'site', 'job_url', 'job_url_direct', 'title', 'company', 'location', 'date_posted', 'job_type', 'salary_source', 'interval', 'min_amount', 'max_amount', 'currency', 'is_remote', 'job_level', 'job_function', 'listing_type', 'emails', 'description', 'company_industry', 'company_url', 'company_logo', 'company_url_direct', 'company_addresses', 'company_num_employees', 'company_revenue', 'company_description', 'skills', 'experience_range', 'company_rating', 'company_reviews_count', 'vacancy_count', 'work_from_home_type'], dtype='object')


In [8]:
# Format JobSpy Data
jobspy_df['job_id'] = jobspy_df['id']
# jobspy_df['location'] = jobspy_df['location'].apply(normalize_location)
jobspy_df['salary_str'] = jobspy_df.apply(format_jobspy_salary, axis=1)
jobspy_df['source'] = jobspy_df['site']
jobspy_df['work_type'] = jobspy_df['job_type'].fillna('')
jobspy_df['work_arrangement'] = jobspy_df.apply(get_work_arrangement, axis=1)
jobspy_df['is_expired'] = False
jobspy_df['country'] = 'SG'

jobspy_df['emails'] = jobspy_df['emails'].apply(clean_emails)


In [9]:
jobspy_df.info(max_cols=None)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id                     100 non-null    object
 1   site                   100 non-null    object
 2   job_url                100 non-null    object
 3   job_url_direct         50 non-null     object
 4   title                  100 non-null    object
 5   company                100 non-null    object
 6   location               100 non-null    object
 7   date_posted            97 non-null     object
 8   job_type               84 non-null     object
 9   salary_source          0 non-null      object
 10  interval               0 non-null      object
 11  min_amount             0 non-null      object
 12  max_amount             0 non-null      object
 13  currency               0 non-null      object
 14  is_remote              100 non-null    bool  
 15  job_level              5

In [33]:
print(jobspy_df.columns)

Index(['id', 'site', 'job_url', 'job_url_direct', 'title', 'company', 'location', 'date_posted', 'job_type', 'salary_source', 'interval', 'min_amount', 'max_amount', 'currency', 'is_remote', 'job_level', 'job_function', 'listing_type', 'emails', 'description', 'company_industry', 'company_url', 'company_logo', 'company_url_direct', 'company_addresses', 'company_num_employees', 'company_revenue', 'company_description', 'skills', 'experience_range', 'company_rating', 'company_reviews_count', 'vacancy_count', 'work_from_home_type', 'job_id', 'salary_str', 'source', 'work_type', 'work_arrangement', 'is_expired', 'country'], dtype='object')


In [14]:
jobspy_df.to_excel('jobspy_output.xlsx', index=False)

## Apify

In [ ]:
print("Scraping JobStreet SG via Apify (blackfalcondata/jobstreet-scraper)...")
apify_raw_data = []
client = ApifyClient(APIFY_TOKEN)
for term in SEARCH_TERMS:
    run = client.actor("blackfalcondata/jobstreet-scraper").call(run_input={
        "query": term,
        "country": "SG",
        "maxResults": 5,          # keep whatever per-term budget you were using, or split your old total across len(SEARCH_TERMS)
        "includeDetails": True,
    })
    apify_raw_data.extend(client.dataset(run.default_dataset_id).list_items().items)


Scraping JobStreet SG via Apify (blackfalcondata/jobstreet-scraper)...


[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> Status: RUNNING, Message: 
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> 2026-09-04T03:34:34.606Z ACTOR: Pulling container image of build lT0OI3UlndEiYtPcM from registry.
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> 2026-09-04T03:34:34.609Z ACTOR: Creating container.
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> 2026-09-04T03:34:34.700Z ACTOR: Starting container.
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> 2026-09-04T03:34:34.702Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> 2026-09-04T03:34:36.461Z INFO  System info {"apifyVersion":"3.7.0","apifyClientVersion":"2.23.1","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v22.23.2"}
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV7p] -> 2026-09-04T03:34:36.953Z INFO  Starting {"sources":1,"maxResults":5,"maxPages":5,"includeDetails":true,"country":"SG"}
[apify.jobstreet-scraper runId:vCEMIQc03CCBehV

In [ ]:
# print(run.default_dataset_id)
# print(f"Check your data here: https://console.apify.com/storage/datasets/{run['defaultDatasetId']}")

In [26]:
apify_df = pd.DataFrame(apify_raw_data)
# apify_df = pd.read_csv("./apify_output.csv")

In [27]:
apify_df['searchQuery'].unique()

array(['data engineer', 'data scientist', 'machine learning engineer',
       'data analyst', 'AI engineer'], dtype=object)

In [28]:
apify_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 73 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   jobId                  45 non-null     object 
 1   seekJobId              45 non-null     object 
 2   title                  45 non-null     object 
 3   canonicalUrl           45 non-null     object 
 4   company                45 non-null     object 
 5   companyUrl             37 non-null     object 
 6   advertiserId           45 non-null     object 
 7   location               45 non-null     object 
 8   locationCountry        45 non-null     object 
 9   locationState          0 non-null      object 
 10  locationSuburb         7 non-null      object 
 11  locationPostcode       0 non-null      object 
 12  salaryText             15 non-null     object 
 13  salaryMin              14 non-null     float64
 14  salaryMax              14 non-null     float64
 15  salaryCu

In [ ]:
print(apify_df['locationCountry'].unique())
apify_df = apify_df[apify_df['locationCountry']=='SG']

apify_df['job_id'] = apify_df['seekJobId']
apify_df['emails'] = apify_df['extractedEmails']

apify_df['salary_str'] = apify_df['salaryText'].fillna('')
apify_df['min_amount'] = apify_df['salaryMin']
apify_df['max_amount'] = apify_df['salaryMax']
apify_df['currency'] = apify_df['salaryCurrency']
apify_df['source'] = 'JobStreet'
apify_df['work_type'] = apify_df['employmentType'].fillna('')
apify_df['is_expired'] = False
apify_df['country'] = 'SG'
apify_df['interval'] = apify_df['salaryType']
apify_df['job_url'] = apify_df['canonicalUrl']
apify_df['company_url'] = apify_df['companyUrl']
apify_df['company_industry'] = apify_df['companyIndustry']

_arrangement_map = {'onsite': 'On-site', 'hybrid': 'Hybrid', 'remote': 'Remote'}
apify_df['work_arrangement'] = apify_df['workArrangement'].str.lower().replace(_arrangement_map).fillna('On-site/Unspecified')

apify_df['postedDate'] = pd.to_datetime(apify_df['postedDate'], utc=True).dt.tz_localize(None)
apify_df['date_posted'] = apify_df['postedDate'].dt.day.astype(str) + '/' + apify_df['postedDate'].dt.month.astype(str) + '/' + apify_df['postedDate'].dt.year.astype(str)

apify_df['jobstreet_job_score'] = apify_df['jobScore']

['SG']


In [30]:
def clean_html(html):
    if not isinstance(html, str): return ''
    return re.sub('<[^<]+?>', '', html)

# Fallback chain: plain description -> markdown -> stripped HTML
apify_df['description'] = apify_df['description'].fillna('')
needs_fallback = apify_df['description'].str.strip() == ''
apify_df.loc[needs_fallback, 'description'] = apify_df.loc[needs_fallback, 'descriptionMarkdown'].fillna('')
needs_fallback = apify_df['description'].str.strip() == ''
apify_df.loc[needs_fallback, 'description'] = apify_df.loc[needs_fallback, 'descriptionHtml'].apply(clean_html)

bullets_str = apify_df['bulletPoints'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x).fillna('')
apify_df['description'] = (apify_df['teaser'].fillna('') + ' ' + apify_df['description'] + ' ' + bullets_str).str.strip()

In [31]:
apify_df['emails'] = apify_df['emails'].apply(clean_emails)

In [33]:
apify_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 90 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   jobId                  45 non-null     object             
 1   seekJobId              45 non-null     object             
 2   title                  45 non-null     object             
 3   canonicalUrl           45 non-null     object             
 4   company                45 non-null     object             
 5   companyUrl             37 non-null     object             
 6   advertiserId           45 non-null     object             
 7   location               45 non-null     object             
 8   locationCountry        45 non-null     object             
 9   locationState          0 non-null      object             
 10  locationSuburb         7 non-null      object             
 11  locationPostcode       0 non-null      object             
 

In [39]:
apify_df.to_excel('apify_output.xlsx',index=False)

## Merged df

In [40]:
def determine_seniority(title):
    """Checks title only and returns a comma-separated set of matches."""
    title_lower = str(title).lower()
    found = set()
    
    if any(w in title_lower for w in ['senior', 'lead', 'principal', 'head', 'manager', 'staff', 'director', 'vp']):
        found.add('Senior/Lead')
    if any(w in title_lower for w in ['mid', 'associate']):
        found.add('Mid')
    if any(w in title_lower for w in ['junior', 'entry', 'graduate', 'trainee', 'fresh']):
        found.add('Entry')
    if any(w in title_lower for w in ['intern']):
        found.add('Intern')
    return ", ".join(found) if found else 'Unknown'

def determine_visa_eligibility(description):
    """Scans for explicit mentions of Singapore work pass quotas and citizenship requirements."""
    if not isinstance(description, str): return 'Unknown'
    desc_lower = description.lower()
    
    local_patterns = [
        r'singaporeans? only', r'singapore citizens?', r'prs? only', 
        r'singaporeans? and prs?', r'singaporeans?/prs?', r'no quota',
        r'sponsorship not available', r'no work pass quota', r'locals only', r'pr preferred'
    ]
    if any(re.search(pat, desc_lower) for pat in local_patterns):
        return 'Local/PR Only'
        
    sponsor_patterns = [
        r'sponsorship available', r'visa sponsorship', r'work pass sponsorship',
        r'quota available', r'ep\s?/\s?sp available', r'employment pass provided'
    ]
    if any(re.search(pat, desc_lower) for pat in sponsor_patterns):
        return 'Sponsorship Available'
        
    return 'Unknown'

def setup_keyword_processors():
    processors = {}
    categories = {
        'is_agent': ['claude', 'gemini', 'cursor', 'langchain', 'llamaindex', 'autogen', 'crewai', 'agentic', 'devin', 'copilot'],
        'is_ai_llm': ['llm', 'generative ai', 'rag', 'fine-tuning', 'prompt engineering', 'transformer', 'vector db', 'openai', 'anthropic'],
        'is_de': ['etl', 'elt', 'hadoop', 'pyspark', 'spark', 'airflow', 'snowflake', 'databricks', 'bigquery', 'kafka', 'dbt'],
        'is_ds': ['machine learning', 'deep learning', 'pytorch', 'tensorflow', 'scikit-learn', 'xgboost', 'nlp'],
        'is_swe': ['backend', 'frontend', 'fullstack', 'rest api', 'microservices', 'docker', 'kubernetes', 'ci/cd', 'fastapi', 'django']
    }
    for col_name, keywords in categories.items():
        kp = KeywordProcessor(case_sensitive=False)
        for kw in keywords: kp.add_keyword(kw)
        processors[col_name] = kp
    return processors

In [ ]:
keep_cols = [
    'job_id', 'title', 'company', 'location', 'source', 'country',
    'salary_str', 'min_amount', 'max_amount', 'currency', 'interval',
    'work_type', 'work_arrangement', 'is_expired',
    'company_url', 'company_industry',
    'emails',
    'job_url', 'job_url_direct',
    'description', 'jobstreet_job_score',
    'date_posted'
]

jobspy_keep = jobspy_df.reindex(columns=keep_cols)
apify_keep = apify_df.reindex(columns=keep_cols)

merged_df = pd.concat([jobspy_keep, apify_keep], ignore_index=True)
merged_df = merged_df.drop_duplicates(subset='job_id', keep='first').reset_index(drop=True)



In [42]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_id               139 non-null    object 
 1   title                139 non-null    object 
 2   company              139 non-null    object 
 3   location             139 non-null    object 
 4   source               139 non-null    object 
 5   country              139 non-null    object 
 6   salary_str           139 non-null    object 
 7   min_amount           14 non-null     float64
 8   max_amount           14 non-null     float64
 9   currency             45 non-null     object 
 10  interval             13 non-null     object 
 11  work_type            139 non-null    object 
 12  work_arrangement     139 non-null    object 
 13  is_expired           139 non-null    bool   
 14  company_url          131 non-null    object 
 15  company_industry     89 non-null     obj

In [54]:
# Description-derived columns
combined_text = merged_df['title'].fillna('') + ' ' + merged_df['description'].fillna('')

merged_df['min_years_exp'] = merged_df['description'].apply(extract_years_exp)
merged_df['seniority'] = merged_df['title'].apply(determine_seniority)
merged_df['visa_eligibility'] = merged_df['description'].apply(determine_visa_eligibility)

keyword_processors = setup_keyword_processors()
for col_name, kp in keyword_processors.items():
    merged_df[col_name] = combined_text.apply(lambda text: bool(kp.extract_keywords(text)))

# Workflow columns — blank/default, filled in manually as you review
merged_df['applied'] = False
merged_df['status'] = 'New'
merged_df['notes'] = ''
merged_df['date_scraped'] = datetime.now().strftime('%Y-%m-%d')

In [48]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_id               20 non-null     object 
 1   title                20 non-null     object 
 2   company              20 non-null     object 
 3   location             20 non-null     object 
 4   source               20 non-null     object 
 5   country              20 non-null     object 
 6   salary_str           20 non-null     object 
 7   min_amount           3 non-null      float64
 8   max_amount           3 non-null      float64
 9   currency             5 non-null      object 
 10  interval             0 non-null      object 
 11  work_type            20 non-null     object 
 12  work_arrangement     20 non-null     object 
 13  is_expired           20 non-null     bool   
 14  company_url          19 non-null     object 
 15  company_industry     9 non-null      objec

In [55]:
merged_df.to_excel(FILE_NAME, index=False)